In [1]:
# 1. Verificacion de entorno y disponibilidad de GPU
import torch
import ultralytics


In [ ]:
# 2. Descarga del conjunto de datos desde Roboflow
from roboflow import Roboflow
import os

rf = Roboflow(api_key="U3hQOadkGWmhw21nSqQO")

project = rf.workspace("g-yezgmez-uandresbello-edu").project("papas-cag92")
version = project.version(1)

dataset = version.download("yolov11")
print(f"\ndataset configurado en: {dataset.location}")

In [ ]:
# 3. Configuracion y Entrenamiento del Modelo YOLOv11
from ultralytics import YOLO

print("\n--- INICIANDO ENTRENAMIENTO ---")
model = YOLO('yolo11n.pt') 

# Iniciar el entrenamiento
resultados = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,      
    imgsz=640,  
    batch=16,              
    device=0,            
    project='Papas_Quemadas',
    name='entrenamiento_1',
    plots=True
)
print("\nEntrenamiento finalizado correctamente.")

In [ ]:
# 4. Evaluacion del Modelo (Validacion)
print("\n--- EVALUACION DE METRICAS ---")
metricas = model.val()
print("\nMetricas de precision generadas y guardadas en el directorio del proyecto.")

In [ ]:
# 5. Demostracion en Imagenes
import glob
import matplotlib.pyplot as plt
import cv2

print("\n--- DEMOSTRACION DE PREDICCIONES ESTATICAS ---")
imagenes_prueba = glob.glob(f"{dataset.location}/valid/images/*.jpg")[:3]

if imagenes_prueba:
    predicciones = model.predict(source=imagenes_prueba, conf=0.5, save=False)

    plt.figure(figsize=(15, 5))
    for i, resultado in enumerate(predicciones):
        imagen_anotada = resultado.plot()
        imagen_rgb = cv2.cvtColor(imagen_anotada, cv2.COLOR_BGR2RGB)
        
        plt.subplot(1, 3, i + 1)
        plt.imshow(imagen_rgb)
        plt.axis('off')
        plt.title(f"Deteccion {i+1}")
    
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron imagenes")

In [ ]:
# 6A. Deteccion en Tiempo Real y Conexion con PLC (Node-RED)
import cv2
import os
import requests # ¡Libreria necesaria para enviar datos a Node-RED!
from ultralytics import YOLO

print("\n--- DETECCION EN VIVO Y COMUNICACION INDUSTRIAL ---")

# ---------------------------------------------------------
# CONFIGURACION DE RUTAS Y RED
# ---------------------------------------------------------
ruta_mejores_pesos = r"C:\Users\sieg\Downloads\Modelo Entrenado\Modelo Entrenado\best.pt"
#fuente_video = "http://10.41.54.65:8080/video"
fuente_video = 0

NODE_RED_URL = "http://127.0.0.1:1880/recibir-vision" # Asegurate que sea la de tu flujo

if not os.path.exists(ruta_mejores_pesos):
    raise FileNotFoundError(f"❌ ERROR: No se encontro el archivo en la ruta:\n{ruta_mejores_pesos}")

print(f"✅ Cargando modelo YOLO desde:\n{ruta_mejores_pesos}")
modelo_tiempo_real = YOLO(ruta_mejores_pesos)

print("\nIniciando camara... espere un momento.")
cap = cv2.VideoCapture(fuente_video)

if not cap.isOpened():
    print("❌ ERROR CRITICO: OpenCV no pudo acceder a la camara.")
else:
    print("✅ Camara conectada con exito!")
    print("👉 PARA SALIR: Haz clic en la ventana de video y presiona la tecla 'q'.\n")
    
    # Variable para recordar el estado y no saturar Node-RED de mensajes repetidos
    estado_anterior = None 
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error de comunicacion con la camara. Cerrando...")
            break
        
        # Prediccion en tiempo real
        resultados_live = modelo_tiempo_real(frame, stream=True, conf=0.45, verbose=False)
        
        # Por defecto, asumimos que no hay papas quemadas en la imagen
        hay_quemada = False 
        
        for r in resultados_live:
            # Dibujar las cajas en el video
            frame_anotado = r.plot()
            cv2.imshow("Detector de Papas - EN VIVO", frame_anotado)
            
            # Analizar qué detectó YOLO en este frame
            # r.boxes.cls contiene una lista con los números de las clases detectadas
            for cls_id in r.boxes.cls:
                nombre_clase = modelo_tiempo_real.names[int(cls_id)]
                
                if nombre_clase == "quemada":
                    hay_quemada = True
                    break # Con una sola que encuentre, ya debe activarse la alarma/rechazo
        
        # ---------------------------------------------------------
        # ENVIO DE SEÑAL AL PLC (VIA NODE-RED)
        # ---------------------------------------------------------
        # Solo enviamos el dato HTTP si el estado cambió respecto al frame anterior
        if hay_quemada != estado_anterior:
            # Mandamos True (1) si hay quemada, False (0) si solo hay sanas
            payload = {"valor": hay_quemada} 
            
            try:
                # Enviamos la petición. Timeout muy corto para que no congele el video
                requests.post(NODE_RED_URL, json=payload, timeout=0.5)
                
                if hay_quemada:
                    print("🔥 ¡ALERTA! Papa quemada detectada -> PLC: True (1)")
                else:
                    print("🥔 Zona limpia / Papas normales -> PLC: False (0)")
                    
            except requests.exceptions.RequestException:
                print("⚠️ [ADVERTENCIA] No se pudo conectar con Node-RED.")
            
            # Actualizamos la memoria
            estado_anterior = hay_quemada
                
        # Control para salir de la ventana
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    # Limpieza final
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("\n✅ Transmision en vivo finalizada y puertos liberados.")

In [ ]:
# 7. Exportacion del modelo
print("\n--- EXPORTACION DEL MODELO ---")

try:
    # Exportamos el modelo recien cargado 
    ruta_exportacion = modelo_tiempo_real.export(format='onnx', imgsz=640)
    print(f"\n\u2705 Exportacion exitosa. Archivo generado: {ruta_exportacion}")
except NameError:
    print("\u274c ERROR: Tienes que ejecutar el Bloque 6 primero para cargar el modelo personalizado antes de exportarlo.")
except Exception as error:
    print(f"Excepcion durante la exportacion: {error}")